# TODAY PONDERS THE QUESTION, Is openai/gpt-oss-120b Actually the right generator Model?

My generator has used `openai/gpt-oss-120b` since ***month 2 day 11***, and looking back at that day's own code, the swap away from `llama-3.3-70b-versatile` was aa real, deliberate experiment, but never a rigorously compared one. Everywhere else in the pipeline (`classify_node`, `review_node`, RAGAS' own `ragas_llm evaluator`) still runs on `llama-3.3-70b-versatile`. ***Day 9*** gives the generator choice the same evidence-first treatment Days 6-7 gave the embedding model.

# Coding: An Isolated Generator Comparison
**Retrieve Once, Generate Three ways:** Unlike ***Days 1-5***, this test does not run the full graph three times per question. Retrieval happens once per golden question (reusing my existing `retrieve_with_confidence`), and the same retrieved context gets handed to three different candidate models for generation. This controls for retrieval variance entirely, any score difference is attributable to the generator alone, not to different searches happening to fire differently each time.

**Three Candidates:** `openai/gpt-oss-120b` (***current***), `llama-3.3-70b-versatile` (***used everywhere else in my pipeline already***), and `openai/gpt-oss-20b` (***the smaller variant***) - worth including specifically because Day 3's real run already showed it throwing `LLMDidNotFinishException` under **RAGAS**' Evaluator role; today checks whether it's also weak as a generator, or if that was specific to it's evaluator duties.

**A fixed Separate Evaluator:** **RAGAS'** judge stays `llama-3.3-70b-versatile` throughout, for all three candidates, including when `llama-3.3-70b-versatile` itself is the generator being judged. This isn't ideal but it's the same evaluator  day 1-4 already validated,
and changing two variables at once ***(generator and judge)*** would make today's result impossible to interprete cleanly.


In [2]:
# ADDING CODE FOR TODAY which is where today's logic starts


from ragas import evaluate
from ragas.metrics import _faithfulness, _answer_correctness, _context_precision
from datasets import Dataset
import pandas as pd

print("👍 RAGAS imported...")

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_cohere import ChatCohere, CohereEmbeddings

import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


ragas_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# setting up embeddings via cohere
cohere_embeddings = CohereEmbeddings(
    model= "embed-english-v3.0",
    cohere_api_key= os.getenv("COHERE_API_KEY")
)

# wrapping models for Ragas compatibility
evaluator_llm = LangchainLLMWrapper(ragas_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(cohere_embeddings)

# configuring RunConfig to eliminate timeout errors
run_config = RunConfig(
    timeout= 180,
    max_workers= 2,
    max_retries = 10,
    max_wait= 60

)

# THE GOLDEN DATASET
# Every answer in this one should be one you've personally checked against the real apple FY2024 10-K before 
# trusting it as ground truth

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5


def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score


def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


GOLDEN_DATASET = [
    {
        "question": "What was Apple's total net sales for fiscal year 2024?",
        "ground_truth": "Apple's total net sales for fiscal year 2024 were $391,035 million."
    },
    {
        "question": "What was Apple's net income for fiscal year 2024?",
        "ground_truth": "Apple's net income for fiscal year 2024 was $93,736 million."
    },
    {
        "question": "What was Apple's total net sales for fiscal year 2023?",
        "ground_truth": "Apple's total net sales for fiscal year 2023 were $383,285 million."
    },
    {
        "question": "What was Apple's iPhone revenue in fiscal year 2024?",
        "ground_truth": "Apple's iPhone net sales for fiscal year 2024 were $201,183 million."
    },
    {
        "question": "What was Apple's Services revenue in fiscal year 2024?",
        "ground_truth": "Apple's Services net sales for fiscal year 2024 were approximately $96,169 million."
    },
    {
        "question": "What was Apple's total operating expenses in fiscal year 2024?",
        "ground_truth": "Apple's total operating expenses for fiscal year 2024 were approximately $57,467 million."
    },
    {
        "question": "What was Apple's cash and cash equivalents at the end of fiscal year 2024?",
        "ground_truth": "Apple's cash and cash equivalents at the end of fiscal year 2024 were $29,943 million."
    },
    {
        "question": "What was Apple's total current liabilities at the end of fiscal year 2024?",
        "ground_truth": "Apple's total current liabilities at the end of fiscal year 2024 were $176,392 million."
    },
    {
        "question": "What fiscal year end date does Apple's FY2024 10-K cover?",
        "ground_truth": "Apple's fiscal year 2024 ended on September 28, 2024."
    },
    {
        "question": "What was Apple's research and development spending in fiscal year 2024?",
        "ground_truth": "Apple's research and development expense for fiscal year 2024 was approximately $31,370 million."
    },
    {
        "question": "By how much did Apple's total net sales grow from fiscal year 2023 to fiscal year 2024?",
        "ground_truth": "Apple's total net sales grew from $383,285 million in FY2023 to $391,035 million in FY2024, an increase of approximately $7,750 million, or about 2%."
    },
    {
        "question": "What was Apple's gross margin percentage approximately in fiscal year 2024?",
        "ground_truth": "Apple's gross margin for fiscal year 2024 was approximately 46.2% of total net sales."
    },
]

print(f"📋 Golden dataset built: {len(GOLDEN_DATASET)} questions")

👍 RAGAS imported...


C:\Users\rodne\AppData\Local\Temp\ipykernel_1688\789096192.py:44: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


🤖🛩️ Vector Store connection established ⚡
📋 Golden dataset built: 12 questions


C:\Users\rodne\AppData\Local\Temp\ipykernel_1688\789096192.py:145: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ragas_llm)
C:\Users\rodne\AppData\Local\Temp\ipykernel_1688\789096192.py:146: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(cohere_embeddings)


In [4]:
# CANDIDATE GENERATORS
CANDIDATE_GENERATORS = {
    "openai/gpt-oss-120b": "Current production generator",
    "openai/gpt-oss-20b": "Smaller variant — struggled as RAGAS's evaluator on Day 3",
}

In [6]:
# RETRIEVE ONCE PER QUESTION 

def retrieve_for_golden_question(question: str) -> tuple[str, list]:
    docs, score = retrieve_with_confidence(question)
    context = format_chunks(docs) if docs else "No relevant content found."
    contexts_list = [c.strip() for c in context.split("---") if c.strip()] or [context]
    return context, contexts_list

retrieved_cache = {}
for item in GOLDEN_DATASET:
    context, contexts_list = retrieve_for_golden_question(item["question"])
    retrieved_cache[item["question"]] = {"context": context, "contexts": contexts_list}

print(f"✅ Retrieved context once for {len(retrieved_cache)} golden questions — "
      f"reused identically across all two candidate generators")

📣 Top retrieval score : 0.866 (threshold: 0.5)
📣 Top retrieval score : 0.867 (threshold: 0.5)
📣 Top retrieval score : 0.877 (threshold: 0.5)
📣 Top retrieval score : 0.855 (threshold: 0.5)
📣 Top retrieval score : 0.860 (threshold: 0.5)
📣 Top retrieval score : 0.881 (threshold: 0.5)
📣 Top retrieval score : 0.870 (threshold: 0.5)
📣 Top retrieval score : 0.859 (threshold: 0.5)
📣 Top retrieval score : 0.864 (threshold: 0.5)
📣 Top retrieval score : 0.865 (threshold: 0.5)
📣 Top retrieval score : 0.879 (threshold: 0.5)
📣 Top retrieval score : 0.885 (threshold: 0.5)
✅ Retrieved context once for 12 golden questions — reused identically across all two candidate generators


In [7]:
# GENERATE WITH A SPECIFIC MODEL, GIVEN FIXED CONTEXT

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

def generate_with_model(model_name: str, question: str, context: str) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0,
    )
    return response.choices[0].message.content

In [8]:
# RUNNING ALL THREE CANDIDTAES, EACH TWICE

def run_generator_comparison_pass(model_name: str, pass_number: int) -> Dataset:
    rows = []
    for item in GOLDEN_DATASET:
        cached = retrieved_cache[item["question"]]
        answer = generate_with_model(model_name, item["question"], cached["context"])
        rows.append({
            "question": item["question"],
            "answer": answer,
            "contexts": cached["contexts"],
            "ground_truth": item["ground_truth"],
        })
    print(f"  ✅ {model_name} — pass {pass_number} generated {len(rows)} answers")
    return Dataset.from_dict({
        "question": [r["question"] for r in rows],
        "answer": [r["answer"] for r in rows],
        "contexts": [r["contexts"] for r in rows],
        "ground_truth": [r["ground_truth"] for r in rows],
    })

run_config = RunConfig(timeout=360, max_workers=2, max_retries=10, max_wait=60)

all_results = {}
for model_name in CANDIDATE_GENERATORS:
    print(f"\n{'='*60}\n{model_name} — {CANDIDATE_GENERATORS[model_name]}\n{'='*60}")
    all_results[model_name] = []
    for pass_num in [1, 2]:
        dataset = run_generator_comparison_pass(model_name, pass_num)
        result = evaluate(
            dataset,
            metrics=[_faithfulness, _answer_correctness, _context_precision],
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
            run_config=run_config,
            raise_exceptions=False,
        )
        all_results[model_name].append(result.to_pandas())


openai/gpt-oss-120b — Current production generator
  ✅ openai/gpt-oss-120b — pass 1 generated 12 answers


Evaluating: 100%|██████████| 36/36 [02:43<00:00,  4.53s/it]


  ✅ openai/gpt-oss-120b — pass 2 generated 12 answers


Evaluating: 100%|██████████| 36/36 [02:57<00:00,  4.93s/it]



openai/gpt-oss-20b — Smaller variant — struggled as RAGAS's evaluator on Day 3
  ✅ openai/gpt-oss-20b — pass 1 generated 12 answers


Evaluating: 100%|██████████| 36/36 [03:05<00:00,  5.16s/it]


  ✅ openai/gpt-oss-20b — pass 2 generated 12 answers


Evaluating: 100%|██████████| 36/36 [02:52<00:00,  4.80s/it]


In [9]:
# REPORTING MEAN, AND Missing rows and the run-to-Run spread

print(f"\n{'='*70}")
print(f"{'Model':<28} {'Pass':>5} {'Faith':>8} {'(n)':>5} {'AnsCorr':>8} {'(n)':>5} {'CtxPrec':>8} {'(n)':>5}")
print("-" * 90)

summary_rows = []
for model_name, passes in all_results.items():
    for i, df in enumerate(passes, 1):
        row = {"model": model_name, "pass": i}
        line = f"{model_name:<28} {i:>5}"
        for metric in ["faithfulness", "answer_correctness", "context_precision"]:
            valid = df[metric].dropna()
            n_missing = len(df) - len(valid)
            mean_val = valid.mean() if len(valid) > 0 else float("nan")
            row[f"{metric}_mean"] = mean_val
            row[f"{metric}_missing"] = n_missing
            line += f" {mean_val:>8.3f} {n_missing:>5}"
        summary_rows.append(row)
        print(line)

print(f"\n{'='*70}\nRUN-TO-RUN SPREAD (Pass 1 vs Pass 2) — a large gap is itself a finding\n{'='*70}")
for model_name in CANDIDATE_GENERATORS:
    p1 = next(r for r in summary_rows if r["model"] == model_name and r["pass"] == 1)
    p2 = next(r for r in summary_rows if r["model"] == model_name and r["pass"] == 2)
    for metric in ["faithfulness_mean", "answer_correctness_mean", "context_precision_mean"]:
        gap = abs(p1[metric] - p2[metric])
        flag = "  ⚠️  UNSTABLE" if gap > 0.15 else ""
        print(f"  {model_name:<28} {metric:<24} gap: {gap:.3f}{flag}")

pd.DataFrame(summary_rows).to_csv("day9_generator_comparison.csv", index=False)
print("\n📁 Saved to day9_generator_comparison.csv")


Model                         Pass    Faith   (n)  AnsCorr   (n)  CtxPrec   (n)
------------------------------------------------------------------------------------------
openai/gpt-oss-120b              1    0.833     0    0.937     0    0.220     0
openai/gpt-oss-120b              2    0.917     0    0.876     0    0.220     0
openai/gpt-oss-20b               1    0.833     0    0.898     0    0.220     0
openai/gpt-oss-20b               2    0.833     0    0.858     0    0.220     0

RUN-TO-RUN SPREAD (Pass 1 vs Pass 2) — a large gap is itself a finding
  openai/gpt-oss-120b          faithfulness_mean        gap: 0.083
  openai/gpt-oss-120b          answer_correctness_mean  gap: 0.062
  openai/gpt-oss-120b          context_precision_mean   gap: 0.000
  openai/gpt-oss-20b           faithfulness_mean        gap: 0.000
  openai/gpt-oss-20b           answer_correctness_mean  gap: 0.040
  openai/gpt-oss-20b           context_precision_mean   gap: 0.000

📁 Saved to day9_generator_compari